# Djinni Round 2 Notebook

Notebook này dùng để xử lý **Round 2** từ file `05_djinni_round1_finals.xlsx`.

## Mục tiêu
- Đọc file Round 1
- Apply kết quả đề xuất từ:
  - `ten_thi_truong` -> `ten_ngoai_thi_truong`
  - `ten_gan_giong` -> `cac_ten_gan_giong`
- Cập nhật trạng thái:
  - `da_kiem = 1`
  - `vong = 2`
  - `buoc = 2`
  - `ghi_chu_djinni = round_2_applied`
- Xuất file mới để tiếp tục kiểm tra / Round 3


In [1]:
from pathlib import Path
import pandas as pd

# ==== CONFIG ====
INPUT_FILE = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/05_djinni_round1_finals.xlsx")
OUTPUT_FILE = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/06_djinni_round2_applied.xlsx")

# Nếu muốn đổi tên file thì chỉ cần sửa 2 dòng trên

In [2]:
# Đọc dữ liệu
df = pd.read_excel(INPUT_FILE)

print("Shape:", df.shape)
print("Columns:")
display(pd.DataFrame({"column": df.columns}))

Shape: (1171, 20)
Columns:


,column
0,nhom_lon
1,nhom_nho
2,cum_ky_nang
3,skill_subgroup
4,ten_goc
5,ten_sach
6,ten_ngoai_thi_truong
7,cac_ten_gan_giong
8,huong_xu_ly
9,ghi_chu_djinni


In [3]:
# Xem nhanh phân bố trước khi xử lý
summary_before = {
    "huong_xu_ly": df["huong_xu_ly"].value_counts(dropna=False),
    "da_kiem": df["da_kiem"].value_counts(dropna=False),
    "vong": df["vong"].value_counts(dropna=False),
    "buoc": df["buoc"].value_counts(dropna=False),
}

for key, value in summary_before.items():
    print(f"\n===== {key} =====")
    display(value.to_frame(name="count"))


===== huong_xu_ly =====


,count
huong_xu_ly,
giu_nguyen,1116
doi_ten,46
them_ten_gan_giong,9



===== da_kiem =====


,count
da_kiem,
0,1171



===== vong =====


,count
vong,
1,1171



===== buoc =====


,count
buoc,
1,1171


## Hàm xử lý Round 2

In [5]:
def normalize_text(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value if value else None


def apply_round2(df_input: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    df = df_input.copy()

    # Chuẩn hóa text trước khi xử lý
    text_cols = [
        "ten_ngoai_thi_truong",
        "cac_ten_gan_giong",
        "ten_thi_truong",
        "ten_gan_giong",
        "ghi_chu_djinni",
        "huong_xu_ly",
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].apply(normalize_text)

    # Điều kiện áp dụng:
    # - nếu có ten_thi_truong -> ghi sang ten_ngoai_thi_truong
    # - nếu có ten_gan_giong -> ghi sang cac_ten_gan_giong
    mask_apply_market_name = df["ten_thi_truong"].notna()
    mask_apply_similar_name = df["ten_gan_giong"].notna()

    # Apply dữ liệu Round 2
    df.loc[mask_apply_market_name, "ten_ngoai_thi_truong"] = df.loc[mask_apply_market_name, "ten_thi_truong"]
    df.loc[mask_apply_similar_name, "cac_ten_gan_giong"] = df.loc[mask_apply_similar_name, "ten_gan_giong"]

    # Cập nhật trạng thái kiểm tra
    df["da_kiem"] = 1
    df["vong"] = 2
    df["buoc"] = 2
    df["ghi_chu_djinni"] = "round_2_applied"

    stats = {
        "tong_so_dong": len(df),
        "so_dong_apply_ten_thi_truong": int(mask_apply_market_name.sum()),
        "so_dong_apply_ten_gan_giong": int(mask_apply_similar_name.sum()),
        "huong_xu_ly": df["huong_xu_ly"].value_counts(dropna=False).to_dict(),
    }

    return df, stats

In [6]:
df_round2, stats = apply_round2(df)

print("Kết quả xử lý:")
for k, v in stats.items():
    print(f"- {k}: {v}")

Kết quả xử lý:
- tong_so_dong: 1171
- so_dong_apply_ten_thi_truong: 1171
- so_dong_apply_ten_gan_giong: 55
- huong_xu_ly: {'giu_nguyen': 1116, 'doi_ten': 46, 'them_ten_gan_giong': 9}


In [7]:
# Xem thử vài dòng sau khi apply
preview_cols = [
    "ten_goc",
    "huong_xu_ly",
    "ten_thi_truong",
    "ten_ngoai_thi_truong",
    "ten_gan_giong",
    "cac_ten_gan_giong",
    "da_kiem",
    "vong",
    "buoc",
    "ghi_chu_djinni",
]

display(df_round2[preview_cols].head(20))

,ten_goc,huong_xu_ly,ten_thi_truong,ten_ngoai_thi_truong,ten_gan_giong,cac_ten_gan_giong,da_kiem,vong,buoc,ghi_chu_djinni
0,Python (computer programming),giu_nguyen,python (computer programming),python (computer programming),None,None,1,2,2,round_2_applied
1,computer vision,giu_nguyen,computer vision,computer vision,None,None,1,2,2,round_2_applied
2,deep learning,giu_nguyen,deep learning,deep learning,None,None,1,2,2,round_2_applied
3,machine learning,them_ten_gan_giong,machine learning,machine learning,"ml, ai, model training","ml, ai, model training",1,2,2,round_2_applied
4,utilise machine learning,giu_nguyen,utilise machine learning,utilise machine learning,None,None,1,2,2,round_2_applied
5,Frostbite (digital game creation systems),giu_nguyen,frostbite (digital game creation systems),frostbite (digital game creation systems),None,None,1,2,2,round_2_applied
6,ICT accessibility standards,giu_nguyen,ict accessibility standards,ict accessibility standards,None,None,1,2,2,round_2_applied
7,advise client on technical possibilities,giu_nguyen,advise client on technical possibilities,advise client on technical possibilities,None,None,1,2,2,round_2_applied
8,analyse big data,giu_nguyen,analyse big data,analyse big data,None,None,1,2,2,round_2_applied
9,application usability,giu_nguyen,application usability,application usability,None,None,1,2,2,round_2_applied


In [8]:
# Lưu file output
df_round2.to_excel(OUTPUT_FILE, index=False)
print(f"Đã lưu file: {OUTPUT_FILE.resolve()}")

Đã lưu file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/06_djinni_round2_applied.xlsx


## Kiểm tra nhanh sau khi lưu

In [9]:
df_check = pd.read_excel(OUTPUT_FILE)

print("Shape output:", df_check.shape)
print("\nPhân bố da_kiem:")
display(df_check["da_kiem"].value_counts(dropna=False).to_frame(name="count"))

print("\nPhân bố vong:")
display(df_check["vong"].value_counts(dropna=False).to_frame(name="count"))

print("\nPhân bố buoc:")
display(df_check["buoc"].value_counts(dropna=False).to_frame(name="count"))

Shape output: (1171, 20)

Phân bố da_kiem:


,count
da_kiem,
1,1171



Phân bố vong:


,count
vong,
2,1171



Phân bố buoc:


,count
buoc,
2,1171


## Gợi ý dùng tiếp cho Round 3
Sau khi chạy xong Round 2, bạn có thể:
1. Lọc các dòng `doi_ten` để soi lại chất lượng tên thị trường.
2. Lọc các dòng `them_ten_gan_giong` để rà alias.
3. Tạo thêm cột review tay nếu muốn QA sâu trước khi chốt final.
